In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/application_train.csv')
print("Dataset loaded:", df.shape)

Dataset loaded: (307511, 122)


In [2]:
# Drop columns with more than 40% missing values
# These columns have too little data to be reliable

threshold = 0.40
missing_pct = df.isnull().mean()

cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Columns with >40% missing : {len(cols_to_drop)}")
print(f"Columns being dropped     :\n{cols_to_drop}")

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nDataset shape after dropping : {df.shape}")

Columns with >40% missing : 49
Columns being dropped     :
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', '

In [3]:
# SK_ID_CURR is just an applicant ID — not useful for prediction
# FLAG_DOCUMENT columns are sparse and low signal

doc_cols = [col for col in df.columns if 'FLAG_DOCUMENT' in col]

cols_to_remove = ['SK_ID_CURR'] + doc_cols

# Only drop if they exist after previous step
cols_to_remove = [c for c in cols_to_remove if c in df.columns]

df.drop(columns=cols_to_remove, inplace=True)
print(f"Removed ID and document flag columns: {len(cols_to_remove)} columns")
print(f"Dataset shape now: {df.shape}")

Removed ID and document flag columns: 21 columns
Dataset shape now: (307511, 52)


In [4]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from numerical list
num_cols = [col for col in num_cols if col != 'TARGET']

print(f"Numerical columns  : {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")

Numerical columns  : 39
Categorical columns: 12

Categorical columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']


In [5]:
# For numerical columns — fill missing with median
# Median is preferred over mean because it is robust to outliers

from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

print("Numerical missing values after imputation:")
print(df[num_cols].isnull().sum().sum(), "missing values remaining")

Numerical missing values after imputation:
0 missing values remaining


In [6]:
# For categorical columns — fill missing with most frequent value

cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Categorical missing values after imputation:")
print(df[cat_cols].isnull().sum().sum(), "missing values remaining")

Categorical missing values after imputation:
0 missing values remaining


In [7]:
# Label encode categorical columns
# We use label encoding because tree-based models handle it well

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Categorical columns encoded successfully")
print(f"Dataset shape: {df.shape}")

Categorical columns encoded successfully
Dataset shape: (307511, 52)


In [8]:
# Creating new meaningful features from existing ones
# These are domain-driven features that a bank analyst would care about

# 1. Credit to Income Ratio — how much credit relative to income
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# 2. Annuity to Income Ratio — monthly repayment burden
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# 3. Credit to Goods Ratio — how much of goods price is financed
df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

# 4. Age in Years — DAYS_BIRTH is negative in this dataset
df['AGE_YEARS'] = abs(df['DAYS_BIRTH']) / 365

# 5. Years Employed — DAYS_EMPLOYED is negative
df['YEARS_EMPLOYED'] = abs(df['DAYS_EMPLOYED']) / 365

# 6. Employment to Age Ratio — how long they have been working vs age
df['EMPLOYMENT_AGE_RATIO'] = df['YEARS_EMPLOYED'] / df['AGE_YEARS']

# 7. Income per Family Member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

print("New features created:")
new_features = [
    'CREDIT_INCOME_RATIO',
    'ANNUITY_INCOME_RATIO',
    'CREDIT_GOODS_RATIO',
    'AGE_YEARS',
    'YEARS_EMPLOYED',
    'EMPLOYMENT_AGE_RATIO',
    'INCOME_PER_PERSON'
]
for f in new_features:
    print(f"  {f}")

print(f"\nFinal dataset shape: {df.shape}")

New features created:
  CREDIT_INCOME_RATIO
  ANNUITY_INCOME_RATIO
  CREDIT_GOODS_RATIO
  AGE_YEARS
  YEARS_EMPLOYED
  EMPLOYMENT_AGE_RATIO
  INCOME_PER_PERSON

Final dataset shape: (307511, 59)


In [9]:
# Some ratios may produce inf if denominator is 0
# Replace inf with NaN then fill with median

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill any new NaN values created by inf replacement
df.fillna(df.median(numeric_only=True), inplace=True)

print("Infinite values handled")
print("Remaining missing values:", df.isnull().sum().sum())

Infinite values handled
Remaining missing values: 0


In [10]:
# Cap outliers at 1st and 99th percentile
# This prevents extreme values from distorting the model
# Called Winsorization — a professional technique

cols_to_cap = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'CREDIT_INCOME_RATIO',
    'YEARS_EMPLOYED'
]

for col in cols_to_cap:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

print("Outliers capped using Winsorization at 1st and 99th percentile")

Outliers capped using Winsorization at 1st and 99th percentile


In [11]:
print("=" * 50)
print("     PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Final Shape          : {df.shape}")
print(f"Missing Values Left  : {df.isnull().sum().sum()}")
print(f"Target Distribution  :")
print(df['TARGET'].value_counts())
print(f"\nDefault Rate         : {df['TARGET'].mean()*100:.2f}%")
print(f"\nNew Features Added   : {len(new_features)}")
print("=" * 50)

     PREPROCESSING SUMMARY
Final Shape          : (307511, 59)
Missing Values Left  : 0
Target Distribution  :
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default Rate         : 8.07%

New Features Added   : 7


In [12]:
# Save the cleaned and engineered dataset
# We will load this directly in Phase 4 for model building

df.to_csv('../data/cleaned_data.csv', index=False)
print("Cleaned dataset saved to ../data/cleaned_data.csv")

Cleaned dataset saved to ../data/cleaned_data.csv
